In [ ]:
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold

DATA = Path("../GDGoC-2026")
TRIASE = DATA / "Triase" / "Triase"
RIWAYAT_DIR = DATA / "Riwayat Medis" / "Riwayat Medis"
KEY, PASIEN = "encounter_id", "patient_id"
TARGET, ID = "target_class", "sample_id"
SEED, N_FOLD, N_REPEAT = 42, 5, 3

GRUP = {}

for folder in (DATA, TRIASE, RIWAYAT_DIR):          # gagal cepat jika path salah
    assert folder.is_dir(), f"Folder tidak ditemukan: {folder.resolve()}"
print("Data dibaca dari:", DATA.resolve())

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
waktu = pd.read_csv(DATA / "index_encounters.csv")
patients = pd.read_csv(DATA / "patients.csv")

print(f"train {train.shape} | test {test.shape} | index_encounters {waktu.shape} | patients {patients.shape}")
print("kolom train:", list(train.columns))
print("kolom test :", list(test.columns))

In [ ]:
vc = train[TARGET].value_counts()                   # jumlah sampel per kelas (hanya dari train, karena label ada di sini)
print(pd.DataFrame({"jumlah": vc, "persen": (vc / len(train) * 100).round(1)}))
print("rasio kelas terbanyak : tersedikit =", round(vc.max() / vc.min(), 1))

tebak_mayoritas = np.full(len(train), vc.idxmax())  # prediktor "bodoh": selalu menjawab kelas terbanyak
print("macro-F1 jika selalu menebak kelas terbanyak:",
      round(f1_score(train[TARGET], tebak_mayoritas, average="macro"), 3))

ax = vc.plot.bar(color="#4C78A8", figsize=(6, 3.2), rot=15)
ax.set_title("Distribusi kelas (train)"); ax.set_ylabel("jumlah"); ax.set_ylim(0, vc.max() * 1.12)
for i, v in enumerate(vc.values):                   # tulis angka di atas tiap batang
    ax.text(i, v + 15, f"{v}", ha="center")
plt.tight_layout(); plt.show()

In [ ]:
inventaris = []
for folder in (DATA, TRIASE, RIWAYAT_DIR):          # telusuri semua CSV di tiga folder
    for f in sorted(folder.glob("*.csv")):
        d = pd.read_csv(f)
        inventaris.append({
            "file": str(f.relative_to(DATA)),             # path relatif terhadap DATA
            "baris": len(d),
            "n_encounter": d[KEY].nunique() if KEY in d.columns else None,   # kunjungan unik (jika tabel punya kunci ini)
            "n_pasien": d[PASIEN].nunique() if PASIEN in d.columns else None,
            "kosong_maks_%": round(d.isna().mean().max() * 100, 1),         # kolom dengan persentase kosong tertinggi
            "kolom": ", ".join(d.columns),
        })
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
inventaris = pd.DataFrame(inventaris)
inventaris

In [ ]:
RIWAYAT = {                                         # tabel riwayat -> nama kolom tanggalnya
    "conditions": "recorded_at", "medications": "recorded_at", "procedures": "recorded_at",
    "immunizations": "recorded_at", "allergies": "recorded_at", "observations": "observed_at",
    "previous_encounter": "history_start",
}

In [ ]:
AUDIT = []                                          # catatan temuan: (temuan, jumlah, penanganan) -> jadi tabel write-up di akhir EDA

# ---------- umur ----------
umur = patients["age_at_visit"]
n_999 = int((umur == 999).sum())                    # 999 = kode "tidak diketahui", bukan umur sungguhan
n_luar = int((umur.notna() & (umur != 999) & ~umur.between(0, 120)).sum())   # nilai tak masuk akal selain 999
print(f"umur: kode 999 = {n_999} | di luar 0-120 selain 999 = {n_luar} | kosong = {umur.isna().sum()}")
print(umur[umur.between(0, 120)].describe().round(1).to_string())
AUDIT.append(("umur kode 999 / di luar 0-120", n_999 + n_luar, "diganti NaN"))

# ---------- income: format campur ----------
inc = patients["income_at_visit"].astype("string").str.strip()
jenis = pd.Series("lainnya", index=inc.index)       # label format; "lainnya" = tidak cocok pola mana pun
POLA = {                                            # pola saling lepas (tidak ada yang tumpang tindih)
    "angka polos (12679.0)": r"\d+(\.\d+)?",
    "singkatan k (115.6k)": r"(?i)[\d.,]+k",
    "dolar ($34,487)": r"\$[\d,.]+",
    "USD (USD 1,234)": r"(?i)usd\s*[\d,.]+",
}
for nama, pola in POLA.items():
    jenis[inc.str.fullmatch(pola).fillna(False).astype(bool)] = nama
jenis[inc.isna()] = "kosong"
print("\nformat income:")
print(pd.DataFrame({"jumlah": jenis.value_counts(), "contoh": inc.groupby(jenis).first()}).to_string())
n_campur = int((~jenis.isin(["kosong", "angka polos (12679.0)"])).sum())
AUDIT.append(("income berformat non-polos (USD/$/k/lainnya)", n_campur, "angka(): buang simbol, 'k' x1000, koma ribuan"))
AUDIT.append(("income kosong", int((jenis == "kosong").sum()), "dibiarkan NaN"))

# ---------- status nikah: aman dipersingkat jadi 1 huruf? ----------
mar = patients["marital_at_visit"].dropna().astype(str)
print("\nmarital (mentah):", mar.value_counts().to_dict())
kata_penuh = mar[mar.str.len() > 1]                 # "Married", "Single", ... (bukan kode 1 huruf)
bentrok = kata_penuh.groupby(kata_penuh.str[0]).nunique()   # huruf awal yang dipakai >1 kata berbeda
print("huruf awal yang bentrok (>1 status berbeda):", bentrok[bentrok > 1].to_dict() or "tidak ada")

# ---------- kolom kategori ----------
KAT = ["gender", "race", "ethnicity", "county"]
print("\nkardinalitas:", patients[KAT].nunique().to_dict())
for c in KAT:
    print(f"{c:10s}", patients[c].value_counts(dropna=False).head(5).to_dict())